In [1]:
# Install required packages (run once)
# !pip install pandas numpy matplotlib seaborn plotly scipy anthropic

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
import json
import os
from datetime import datetime, timedelta
from scipy import stats

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 12)

print(" All libraries imported successfully")
print(f" Analysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}") 

 All libraries imported successfully
 Analysis Date: 2026-05-21 20:45:03


In [ ]:
# ─────────────────────────────────────────────
# DATA COLLECTION MODULE
# Source 1: Stock Prices (GBM Simulation)
# ─────────────────────────────────────────────

def simulate_gbm_prices(S0: float, mu: float, sigma: float,
                         n_days: int, seed: int = None) -> np.ndarray:
    """
    Simulate stock prices using Geometric Brownian Motion.
    
    Parameters:
    -----------
    S0     : Initial stock price
    mu     : Annual drift (expected return)
    sigma  : Annual volatility
    n_days : Number of trading days to simulate
    seed   : Random seed for reproducibility
    
    Returns:
    --------
    np.ndarray of simulated daily closing prices
    
    Mathematical basis:
        S(t+dt) = S(t) * exp((mu - 0.5*sigma^2)*dt + sigma*sqrt(dt)*Z)
        where Z ~ N(0,1) (standard normal innovation)
    """
    if seed is not None:
        np.random.seed(seed)
    
    dt = 1 / 252  # one trading day as fraction of year
    
    # Daily log-returns from GBM
    log_returns = (mu - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * np.random.standard_normal(n_days)
    
    # Cumulative product to get price path
    price_path = S0 * np.exp(np.cumsum(log_returns))
    return price_path


def generate_ohlcv(close_prices: np.ndarray, seed: int = 42) -> pd.DataFrame:
    """
    Generate realistic OHLCV data from a close price series.
    Open/High/Low are derived from close with realistic noise.
    Volume follows a log-normal distribution with mean-reversion.
    
    Parameters:
    -----------
    close_prices : Array of closing prices
    seed         : Random seed
    
    Returns:
    --------
    DataFrame with Open, High, Low, Close, Volume columns
    """
    rng = np.random.default_rng(seed)
    n = len(close_prices)
    
    noise = rng.uniform(0.005, 0.015, n)  # intraday range 0.5–1.5%
    
    open_  = close_prices * (1 + rng.uniform(-0.008, 0.008, n))
    high   = close_prices * (1 + noise)
    low    = close_prices * (1 - noise)
    volume = rng.lognormal(mean=np.log(1_000_000), sigma=0.5, size=n).astype(int)
    
    return pd.DataFrame({
        'Open':   open_,
        'High':   high,
        'Low':    low,
        'Close':  close_prices,
        'Volume': volume
    })


# ── Asset Parameters ───────────────────────────────────────────────────────────
# Calibrated to approximate historical S&P 500 constituents (2020–2024 avg)
ASSET_PARAMS = {
    'AAPL': {'S0': 150.0, 'mu': 0.25, 'sigma': 0.28, 'sector': 'Technology'},
    'MSFT': {'S0': 300.0, 'mu': 0.22, 'sigma': 0.24, 'sector': 'Technology'},
    'JPM':  {'S0': 140.0, 'mu': 0.12, 'sigma': 0.22, 'sector': 'Financials'},
    'XOM':  {'S0': 100.0, 'mu': 0.08, 'sigma': 0.25, 'sector': 'Energy'},
    'GLD':  {'S0': 180.0, 'mu': 0.05, 'sigma': 0.15, 'sector': 'Commodities'},
}

# Trading calendar: 2 years of business days (Jan 2023 – Dec 2024)
TRADING_DATES = pd.bdate_range(start='2023-01-01', end='2024-12-31')
N_DAYS = len(TRADING_DATES)

# ── Generate Price Data ────────────────────────────────────────────────────────
stock_data = {}
ohlcv_data = {}

for i, (ticker, params) in enumerate(ASSET_PARAMS.items()):
    prices = simulate_gbm_prices(
        S0    = params['S0'],
        mu    = params['mu'],
        sigma = params['sigma'],
        n_days= N_DAYS,
        seed  = 42 + i  # different seed per asset, but reproducible
    )
    stock_data[ticker] = prices
    ohlcv_data[ticker] = generate_ohlcv(prices, seed=100+i)
    ohlcv_data[ticker].index = TRADING_DATES

# Main price DataFrame
prices_df = pd.DataFrame(stock_data, index=TRADING_DATES)
prices_df.index.name = 'Date'

print("=" * 60)
print(f"📊 Stock Price Data Collected")
print("=" * 60)
print(f"  Assets     : {list(prices_df.columns)}")
print(f"  Period     : {prices_df.index[0].date()} → {prices_df.index[-1].date()}")
print(f"  Trading days: {len(prices_df):,}")
print(f"  Data shape : {prices_df.shape}")
print()
print("  Starting Prices:")
for ticker, params in ASSET_PARAMS.items():
    print(f"    {ticker:5s}: ${params['S0']:.2f}  (μ={params['mu']:.0%}, σ={params['sigma']:.0%})")
print()
print(prices_df.tail(5).to_string())